In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns",None)

In [2]:
df = pd.read_csv("../data/creditcard.csv")
print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
print (df["Class"].value_counts)
print()
print(df["Class"].value_counts(normalize=True) * 100)


<bound method IndexOpsMixin.value_counts of 0         0
1         0
2         0
3         0
4         0
         ..
284802    0
284803    0
284804    0
284805    0
284806    0
Name: Class, Length: 284807, dtype: int64>

Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


print (df.isnull().sum().sum())
print(df["Amount"].describe()]

In [4]:
from sklearn.preprocessing import StandardScaler
import joblib

# Rebuild X fresh to avoid double-scaling
X = df.drop("Class", axis=1)
y = df["Class"]

scaler = StandardScaler()
X[["Time", "Amount"]] = scaler.fit_transform(X[["Time", "Amount"]])

joblib.dump(scaler, "../model/scaler.pkl")
print("Saved. Features:", scaler.feature_names_in_)


Saved. Features: ['Time' 'Amount']


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# rerun your XGBoost fit cell here, then:X
joblib.dump(model_xgb, "../model/fraud_model.pkl")
print("Model re-saved")

NameError: name 'model_xgb' is not defined

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


# using the sigmoid function. We then threshold at 0.5 to predict the class.
model = LogisticRegression(max_iter=1000)

# "Training" = fitting parameters (w, b) by minimizing the cost function
model.fit(X_train, y_train)

# Predictions on the test set the model has never seen
y_pred = model.predict(X_test)

# Probabilities (the sigmoid output) for the fraud class
y_prob = model.predict_proba(X_test)[:, 1]

# Confusion matrix: [[TN, FP], [FN, TP]]
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Precision & Recall (Andrew Ng's skewed-data lecture)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
model_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
model_balanced.fit(X_train, y_train)

y_pred_bal = model_balanced.predict(X_test)
y_prob_bal = model_balanced.predict_proba(X_test)[:, 1]

print("Confusion Matrix (balanced):")
print(confusion_matrix(y_test, y_pred_bal))
print("\nClassification Report (balanced):")
print(classification_report(y_test, y_pred_bal))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_bal))

In [ ]:
from xgboost import XGBClassifier

# scale_pos_weight tells XGBoost how much to weight the rare fraud class
# (ratio of normal to fraud ≈ 577)
model_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=577,
    eval_metric="logloss",
    random_state=42
)
model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
y_prob_xgb = model_xgb.predict_proba(X_test)[:, 1]

print("Confusion Matrix (XGBoost):")
print(confusion_matrix(y_test, y_pred_xgb))
print("\nClassification Report (XGBoost):")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

In [ ]:
import shap

X_sample = X_test.sample(2000, random_state=42)
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=True)

In [ ]:
import joblib

# Save the trained XGBoost model
joblib.dump(model_xgb, "../model/fraud_model.pkl")

# Save the scaler too 
joblib.dump(scaler, "../model/scaler.pkl")

print("Model and scaler saved to model/ folder")


In [ ]:
import joblib
s = joblib.load("../model/scaler.pkl")
print(s.feature_names_in_)
print(s.n_features_in_)